In [1]:
RANDOM_STATE = 42

from pathlib import Path
import sys

BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

sys.path.insert(0, str(BASE_DIR))

# 02 — Pré-processamento

## Objetivo

Esta etapa transforma a análise exploratória em dados adequados para modelagem. O processo mantém o conjunto de teste isolado, documenta o tratamento de valores ausentes, aplica padronização quando necessária e avalia se novas variáveis derivadas agregam valor preditivo.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from src.config import DATA_FILE, PROCESSED_DATA_DIR, RANDOM_STATE
from src.data import load_dataset
from src.preprocessing import create_target, get_features, add_engineered_features

df = create_target(load_dataset(DATA_FILE))
features = get_features(df)
X = df[features].copy()
y = df["high_quality"].copy()

## 1. Separação treino/teste

O conjunto de dados é separado em 80% para treino e 20% para teste com estratificação pela variável alvo. O teste permanece intocado até a etapa de avaliação final.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (914, 11)
Test set: (229, 11)


### Interpretação

A estratificação preserva aproximadamente a proporção das duas classes nos conjuntos de treino e teste. Nenhuma transformação aprendida é ajustada sobre o conjunto de teste.

## 2. Missing values e normalização

O dataset não apresenta valores ausentes. Mesmo assim, a imputação pela mediana é mantida dentro do pipeline como mecanismo de robustez para futuras observações.

A padronização é aplicada à Regressão Logística porque modelos lineares são sensíveis à escala dos atributos. Para Random Forest e Gradient Boosting, a escala não altera a regra de divisão das árvores e, portanto, não é necessária.

In [4]:
logistic_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
logistic_preprocess.fit(X_train)
print("Training matrix after preprocessing:", logistic_preprocess.transform(X_train).shape)

Training matrix after preprocessing: (914, 11)


### Interpretação

A padronização é aplicada apenas onde possui justificativa metodológica. Como as transformações estão em um `Pipeline`, seus parâmetros são aprendidos somente com dados de treino, reduzindo o risco de data leakage.

## 3. Feature engineering

Foram testadas quatro features derivadas com motivação físico-química:

- `total_acidity`;
- `bound_sulfur_dioxide`;
- `free_sulfur_ratio`;
- `sulphates_chlorides_ratio`.

A versão com features derivadas é comparada à versão original pelo F1-score em cinco folds estratificados, utilizando somente o conjunto de treino.

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1","roc_auc":"roc_auc"}

X_fe = add_engineered_features(X)
X_fe_train, _, y_fe_train, _ = train_test_split(
    X_fe, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

rf_base = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)),
])
rf_fe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)),
])

base_scores = cross_validate(rf_base, X_train, y_train, cv=cv, scoring="f1", n_jobs=1)["test_score"]
fe_scores = cross_validate(rf_fe, X_fe_train, y_fe_train, cv=cv, scoring="f1", n_jobs=1)["test_score"]

feature_engineering_comparison = pd.DataFrame({
    "Version": ["Original features", "With engineered features"],
    "Mean F1": [base_scores.mean(), fe_scores.mean()],
    "Std F1": [base_scores.std(), fe_scores.std()],
})
display(feature_engineering_comparison.round(4))
feature_engineering_comparison.to_csv(PROCESSED_DATA_DIR / "feature_engineering_comparison.csv", index=False)

,Version,Mean F1,Std F1
0,Original features,0.5609,0.0494
1,With engineered features,0.5122,0.1038


### Interpretação e decisão

A comparação empírica determina a decisão final. A versão com maior F1 médio na validação cruzada é considerada a melhor alternativa. Se as features derivadas não superarem o conjunto original, o modelo final permanece com as variáveis originais, evitando complexidade sem benefício preditivo.

In [6]:
from src.data import save_dataset
save_dataset(X_train, PROCESSED_DATA_DIR / "X_train.csv")
save_dataset(X_test, PROCESSED_DATA_DIR / "X_test.csv")
save_dataset(y_train.to_frame("high_quality"), PROCESSED_DATA_DIR / "y_train.csv")
save_dataset(y_test.to_frame("high_quality"), PROCESSED_DATA_DIR / "y_test.csv")
print("Processed datasets generated successfully.")

Processed datasets generated successfully.
